# Playlist EDA — listen-wiseer

Covers:
1. Playlist track counts & audio feature heatmap
2. Stacked bars: playlist x gen_4 and gen_8
3. Pairplot by playlist group
4. Outlier analysis (z-score from playlist centroid)
5. TSNE per playlist

In [ ]:
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))

import warnings

warnings.filterwarnings("ignore")
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
DB = str(ROOT / "data/listen_wiseer.db")
con = duckdb.connect(DB, read_only=True)

In [ ]:
from utils.const import playlist_group_dict

tp = con.execute("SELECT * FROM track_profile").df()

pt = con.execute("""
    SELECT pt.track_id, p.playlist_name
    FROM playlist_tracks pt
    JOIN playlists p USING (playlist_id)
""").df()

df = tp.merge(pt, on="track_id", how="inner")
print(f"Playlist-track rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

## 1. Playlist track counts & audio feature heatmap

In [ ]:
playlist_counts = con.execute("""
    SELECT p.playlist_name, p.gen_4, COUNT(pt.track_id) AS track_count
    FROM playlist_tracks pt
    JOIN playlists p USING (playlist_id)
    GROUP BY p.playlist_name, p.gen_4
    ORDER BY track_count DESC
""").df()

fig, ax = plt.subplots(figsize=(10, 5))
palette = sns.color_palette("muted", n_colors=playlist_counts["gen_4"].nunique())
gen4_colors = {g: c for g, c in zip(playlist_counts["gen_4"].unique(), palette, strict=False)}
colors = [gen4_colors.get(g, "gray") for g in playlist_counts["gen_4"]]
ax.barh(playlist_counts["playlist_name"], playlist_counts["track_count"], color=colors)
ax.set_title("Tracks per playlist (colored by gen_4)")
ax.invert_yaxis()
plt.tight_layout()

In [ ]:
playlist_audio = con.execute("""
    SELECT
        p.playlist_name,
        p.gen_4,
        AVG(af.danceability)     AS danceability,
        AVG(af.energy)           AS energy,
        AVG(af.valence)          AS valence,
        AVG(af.acousticness)     AS acousticness,
        AVG(af.instrumentalness) AS instrumentalness,
        AVG(af.tempo)            AS tempo
    FROM playlist_tracks pt
    JOIN playlists p USING (playlist_id)
    JOIN audio_features af USING (track_id)
    GROUP BY p.playlist_name, p.gen_4
    ORDER BY p.gen_4, p.playlist_name
""").df()

pivot = playlist_audio.set_index("playlist_name").drop(columns="gen_4")
fig, ax = plt.subplots(figsize=(10, len(pivot) * 0.45 + 1))
sns.heatmap(
    pivot.apply(lambda c: (c - c.min()) / (c.max() - c.min())),
    annot=pivot.round(2),
    fmt=".2f",
    cmap="RdYlGn",
    linewidths=0.4,
    ax=ax,
)
ax.set_title("Mean audio features per playlist (min-max scaled)")
plt.tight_layout()

## 2. Stacked bars: playlist x genre

In [ ]:
stacked_gen4 = df.groupby("playlist_name")["gen_4"].value_counts().unstack(fill_value=0)
stacked_gen4.plot(kind="barh", stacked=True, figsize=(10, 8))
plt.title("Playlist composition by gen_4")
plt.tight_layout()

In [ ]:
stacked_gen8 = df.groupby("playlist_name")["gen_8"].value_counts().unstack(fill_value=0)
stacked_gen8.plot(kind="barh", stacked=True, figsize=(10, 8))
plt.title("Playlist composition by gen_8")
plt.tight_layout()

## 3. Pairplot by playlist group

In [ ]:
df["playlist_group"] = df["playlist_name"].map(
    lambda x: next((k for k, v in playlist_group_dict.items() if x in v), None)
)

pairplot_cols = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "tempo",
    "playlist_group",
]
pg_df = df[pairplot_cols].dropna(subset=["playlist_group"])
g = sns.pairplot(pg_df, hue="playlist_group", plot_kws={"alpha": 0.3, "s": 10})
g.fig.suptitle("Pairplot by playlist group", y=1.01)
plt.tight_layout()

## 4. Outlier analysis

In [ ]:
audio_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "popularity",
]


def compute_outlier_scores(df, audio_cols):
    """
    For each track, compute Euclidean z-score distance from its playlist centroid
    on the given audio features. Returns df with an added 'score' column.
    """
    df = df.dropna(subset=audio_cols).copy()
    centroids = df.groupby("playlist_name")[audio_cols].mean()

    rows = []
    for _, row in df.iterrows():
        centroid = centroids.loc[row["playlist_name"]]
        diff = row[audio_cols].values - centroid.values
        # z-score: divide by playlist std, then compute norm
        std = df[df["playlist_name"] == row["playlist_name"]][audio_cols].std().replace(0, 1)
        score = np.linalg.norm(diff / std.values)
        rows.append(score)
    df["score"] = rows
    return df


outliers = compute_outlier_scores(df, audio_cols)
outliers[["track_name", "playlist_name", "score"]].sort_values("score", ascending=False).head(20)

In [ ]:
plt.figure(figsize=(14, 6))
sns.histplot(outliers, x="score", hue="playlist_name", kde=True, multiple="stack")
plt.title("Outlier score distribution by playlist")
plt.tight_layout()

In [ ]:
playlists = outliers["playlist_name"].unique()
n = len(playlists)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
for ax, pl in zip(axes.flatten(), playlists, strict=False):
    sns.histplot(outliers[outliers["playlist_name"] == pl]["score"], kde=True, ax=ax, bins=20)
    ax.set_title(pl, fontsize=9)
    ax.set_xlabel("score")
for ax in axes.flatten()[n:]:
    ax.set_visible(False)
fig.suptitle("Outlier score per playlist", fontsize=13)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(
    outliers["energy"],
    outliers["danceability"],
    c=outliers["score"],
    cmap="YlOrRd",
    alpha=0.6,
    s=15,
)
plt.colorbar(sc, ax=ax, label="outlier score")
ax.set_xlabel("energy")
ax.set_ylabel("danceability")
ax.set_title("Energy vs Danceability — colored by outlier score")
plt.tight_layout()

## 5. TSNE per playlist

In [ ]:
from sklearn.manifold import TSNE as _TSNE


def calculate_tsne(df, playlist, audio_cols=audio_cols):
    data = df[df["playlist_name"] == playlist].copy()
    data = data.dropna(subset=audio_cols)
    perplexity = min(30, len(data) - 1)
    if perplexity < 2:
        print(f"Skipping {playlist!r}: only {len(data)} samples after dropna")
        return data
    tsne = _TSNE(n_components=2, random_state=42, perplexity=perplexity)
    X_tsne = tsne.fit_transform(data[audio_cols])
    data["X_tsne"] = X_tsne[:, 0]
    data["y_tsne"] = X_tsne[:, 1]
    data["tsne_group"] = (data["X_tsne"] >= 0).astype(int).map({0: "left", 1: "right"})
    return data


def plot_tsne_scatter(data, axes):
    sns.scatterplot(
        x="X_tsne", y="y_tsne", hue="tsne_group", data=data, palette="viridis", ax=axes[0]
    )


def plot_tsne_by_genres(data, axes):
    for side, color in [("left", "purple"), ("right", "orange")]:
        subset = data[data["tsne_group"] == side]
        if len(subset) == 0:
            continue
        df_counts = subset.groupby("first_genre").size().reset_index(name="Count")
        sns.barplot(x="Count", y="first_genre", data=df_counts, orient="h", color=color, ax=axes[1])

In [ ]:
available = sorted(df["playlist_name"].unique())
print(available)

playlist = available[0]  # use first available to verify TSNE works
tsne_data = calculate_tsne(df, playlist)
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 5))
plot_tsne_scatter(tsne_data, axes)
axes[0].set_title(f"TSNE — {playlist}")
plot_tsne_by_genres(tsne_data, axes)
axes[1].set_title(f"Genre breakdown by TSNE side — {playlist}")
plt.tight_layout()
plt.show()

In [ ]:
con.close()
print("Done.")